# AI-vs-Human lexical baseline (TF-IDF + Naive Bayes) on PolitiFact++ & GossipCop++

Applies the **same technique** as `ai-generated-vs-human-text-95-accuracy.ipynb` to the two
LIFE benchmarks. **Task = provenance:** LLM-generated (MF + MR) = **AI (1)** vs human-written
(HF + HR) = **human (0)**.

- Pure lexical bag-of-words: strip `\n`/`'` → drop punctuation → remove stopwords →
  `CountVectorizer → TfidfTransformer → MultinomialNB`.
- **CPU only** — no GPU needed (Runtime → Change runtime type → CPU is fine). Independent of the
  LIFE fingerprint pipeline; reads the raw article text directly.
- **How to read it:** with a ~40/60 AI/human class balance, **macro-F1 and AI(1) recall** are the
  honest metrics — plain accuracy can look inflated by the majority (human) class.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PROJECT_DIR    = '/content/drive/MyDrive/LIFE'
DATASET_ROOT   = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset'
POLITIFACT_DIR = f'{DATASET_ROOT}/PolitiFact++'
GOSSIPCOP_DIR  = f'{DATASET_ROOT}/GossipCop++'

os.chdir(PROJECT_DIR)  # so the relative script path below resolves
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))
print('GossipCop++  found:', os.path.isdir(GOSSIPCOP_DIR))

PolitiFact++ found: True
GossipCop++  found: True


## Run the baseline

Each cell prints the class balance, accuracy, and a per-class classification report.
sklearn / pandas / nltk are preinstalled on Colab; the script downloads the needed NLTK data on
first run. GossipCop++ (~20k articles) still finishes in well under a minute on CPU.

In [3]:
!python ai_vs_human_code/run_life_baseline.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++

[PolitiFact++] 520 articles | human(0)=291 AI(1)=229

=== PolitiFact++: TF-IDF + MultinomialNB (AI-vs-Human) ===
Accuracy: 0.6474
              precision    recall  f1-score   support

    human(0)       0.62      0.99      0.76        89
       AI(1)       0.93      0.19      0.32        67

    accuracy                           0.65       156
   macro avg       0.77      0.59      0.54       156
weighted avg       0.75      0.65      0.57       156



In [4]:
!python ai_vs_human_code/run_life_baseline.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++

[GossipCop++] 20505 articles | human(0)=12252 AI(1)=8253

=== GossipCop++: TF-IDF + MultinomialNB (AI-vs-Human) ===
Accuracy: 0.7188
              precision    recall  f1-score   support

    human(0)       0.68      1.00      0.81      3656
       AI(1)       1.00      0.31      0.47      2496

    accuracy                           0.72      6152
   macro avg       0.84      0.65      0.64      6152
weighted avg       0.81      0.72      0.67      6152



## Notes
- **Labels are provenance, not veracity:** AI = MF/MR (GPT-3.5), human = HF/HR. To try other
  cuts (fake-vs-real, or LIFE's MF-vs-MR), edit `FILE_LABELS` in `run_life_baseline.py`.
- This baseline uses **no** LIFE machinery (no key sentences, no LLaMA perplexity fingerprints).
  It's a cheap lexical reference point: how much of the AI-vs-human signal already lives in the
  surface vocabulary. Where it falls short is where LIFE's fingerprint method earns its keep.
- Reproducibility: fixed `--seed 42`, `--test_size 0.3` (same as the Kaggle notebook).